In [ ]:
# @title 🔧 Smart Setup — Run once per session
# @markdown **First run (~20 min):** installs envs + downloads weights, saves weights to Drive.
# @markdown ** if run later loads weights from Drive if already present there, skips download.
# @markdown If runtime restarts after condacolab — just re-run this cell once connect the google drive and if prompted to restart session cancel it.

from IPython.utils import io
from IPython.display import HTML, display

print("⏳ Running setup and installations (this may take several minutes)...")

with io.capture_output() as captured:
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 1 — condacolab
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    try:
        import condacolab
        condacolab.check()
        print("✅ condacolab already active")
    except:
        print("⏳ Installing condacolab — runtime will restart automatically.")
        print("   Re-run this cell once after the restart.")
        !pip install -q condacolab
        import condacolab
        condacolab.install()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 2 — mamba (skip if already installed)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    import subprocess, shutil, os
    if shutil.which("mamba"):
        print("✅ mamba already installed")
    else:
        print("⏳ Installing mamba...")
        !conda install -c conda-forge mamba -y
        print("✅ mamba ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 3 — Mount Drive + create folder structure
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE    = '/content/drive/MyDrive/try_pipeline'
    DRIVE_WEIGHTS = f'{DRIVE_BASE}/weights'
    DRIVE_AF2     = f'{DRIVE_WEIGHTS}/colabfold_params'

    os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
    os.makedirs(DRIVE_AF2,     exist_ok=True)
    print("✅ Drive folders ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 4 — Clone repo (skip if already present)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    REPO_DIR = '/content/batch_peptide_pipeline'
    if os.path.isdir(REPO_DIR):
        print("✅ Repo already cloned — skipping git clone")
    else:
        !git clone https://github.com/varun-bhai/batch_peptide_pipeline.git
        print("✅ Repo cloned")

    %cd /content/batch_peptide_pipeline
    !mkdir -p cache colabfold_params
    !rm -rf output

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 5 — Build environments (skip if already exist)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    def env_exists(name):
        r = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
        return name in r.stdout

    if env_exists('etflow_env'):
        print("✅ etflow_env already exists — skipping")
    else:
        print("⏳ Building etflow_env (~3 min)...")
        !mamba create -n etflow_env python=3.10 -y
        !mamba run -n etflow_env pip install requests rdkit biopython
        !mamba run -n etflow_env pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
        !mamba run -n etflow_env pip install torch-cluster==1.6.3 -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
        !mamba run -n etflow_env pip install etflow
        print("✅ etflow_env done")

    if env_exists('af2_env'):
        print("✅ af2_env already exists — skipping")
    else:
        print("⏳ Building af2_env (~3 min)...")
        !mamba create -n af2_env python=3.10 -y
        !mamba run -n af2_env pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
        !mamba run -n af2_env pip install --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
        print("✅ af2_env done")

    if env_exists('mace_env'):
        print("✅ mace_env already exists — skipping")
    else:
        print("⏳ Building mace_env (~2 min)...")
        !mamba create -n mace_env python=3.10 -y
        !mamba run -n mace_env pip install --upgrade torch torchvision torchaudio "numpy<2" --index-url https://download.pytorch.org/whl/cu118
        !mamba run -n mace_env pip install mace-torch ase openmm
        !mamba run -n mace_env pip install git+https://github.com/openmm/pdbfixer.git
        print("✅ mace_env done")

    !pip install biopython -q
    os.environ['MPLBACKEND'] = 'Agg'
    print("✅ All environments ready")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 6 — Weights (load from Drive if cached, else download + save)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    LOCAL_CACHE  = '/content/batch_peptide_pipeline/cache'
    LOCAL_AF2    = '/content/batch_peptide_pipeline/colabfold_params'
    ETFLOW_DRIVE = f'{DRIVE_WEIGHTS}/drugs-o3.ckpt'
    ETFLOW_LOCAL = f'{LOCAL_CACHE}/drugs-o3.ckpt'

    # -- ETFlow checkpoint --
    if os.path.exists(ETFLOW_DRIVE) and os.path.getsize(ETFLOW_DRIVE) > 50_000_000:
        print("\n⏳ ETFlow weights found on Drive — copying locally...")
        shutil.copy(ETFLOW_DRIVE, ETFLOW_LOCAL)
        print("✅ ETFlow weights ready (from Drive)")
    else:
        print("\n⏳ Downloading ETFlow weights from Zenodo (~101 MB)...")
        !wget --continue --tries=10 --timeout=30 -O {ETFLOW_LOCAL} "https://zenodo.org/records/14226681/files/drugs-o3.ckpt?download=1"
        print("💾 Saving ETFlow weights to Drive for future sessions...")
        shutil.copy(ETFLOW_LOCAL, ETFLOW_DRIVE)
        print("✅ ETFlow weights downloaded and saved to Drive")

    # -- AlphaFold params --
    COLABFOLD_CACHE = os.path.join(os.path.expanduser('~'), '.cache', 'colabfold', 'params')
    af2_drive_files = os.listdir(DRIVE_AF2) if os.path.exists(DRIVE_AF2) else []
    if len(af2_drive_files) > 5:
        print("\n⏳ AlphaFold params found on Drive — copying to cache (may take a few min)...")
        os.makedirs(COLABFOLD_CACHE, exist_ok=True)
        for fname in os.listdir(DRIVE_AF2):
            src = os.path.join(DRIVE_AF2, fname)
            dst = os.path.join(COLABFOLD_CACHE, fname)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
        print("✅ AlphaFold params ready (from Drive)")
    else:
        print("\n⏳ Downloading AlphaFold params (~1.8 GB)...")
        !mamba run -n af2_env python -m colabfold.download
        print("💾 Saving AlphaFold params to Drive for future sessions...")
        os.makedirs(DRIVE_AF2, exist_ok=True)
        if os.path.exists(COLABFOLD_CACHE):
            for fname in os.listdir(COLABFOLD_CACHE):
                src = os.path.join(COLABFOLD_CACHE, fname)
                dst = os.path.join(DRIVE_AF2, fname)
                if not os.path.exists(dst):
                    shutil.copy(src, dst)
            print("✅ AlphaFold params downloaded and saved to Drive")
        else:
            print("⚠️ Cache folder not found after download — params may not have saved to Drive")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 7 — Disable explosion triggers in 05_minimize.py
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    script = "/content/batch_peptide_pipeline/05_minimize.py"
    with open(script) as f:
        code = f.read()

    patched = code.replace(
        '    detector = ExplosionDetector(atoms, e0, pos0, check_every=50)\n    opt.attach(detector)',
        '    # detector = ExplosionDetector(atoms, e0, pos0, check_every=50)\n    # opt.attach(detector)'
    )

    with open(script, "w") as f:
        f.write(patched)

    if patched != code:
        print("✅ Explosion triggers disabled in 05_minimize.py")
    else:
        print("✅ Explosion triggers already disabled (no change needed)")

    print("\n🎉 Setup complete! Run Cell 2 for single sequence or Cell 3 for batch.")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 8 — Display Output Toggle
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
display(HTML(f"""
<details>
    <summary style="cursor: pointer; color: #1a73e8; font-weight: bold; padding: 10px; border: 1px solid #1a73e8; border-radius: 5px; width: fit-content;">
        ✅ Setup Complete. Click here to view detailed installation logs.
    </summary>
    <div style="background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; margin-top: 10px; max-height: 500px; overflow-y: auto;">
        <pre style="white-space: pre-wrap; word-wrap: break-word;">{captured.stdout}</pre>
        <pre style="color: red; white-space: pre-wrap; word-wrap: break-word;">{captured.stderr}</pre>
    </div>
</details>
"""))

⏳ Running setup and installations (this may take several minutes)...


In [ ]:
 # @title ▶️ Single Sequence Prediction
# @markdown Enter your sequence and job name below, then hit run.
# @markdown Output folder will be auto-downloaded as a zip when done.
# @markdown (sequence format(typical pdb format): standard AAs + NCAA in brackets e.g. MKA(NVA)AA) or you can also enter the sequence in MAP format for more information on it refer to this paper: https://arxiv.org/abs/2505.03403 the output is stored in google drive folder named: try_pipeline and also as downloaded automatically as zip file if you wish to download all the files made in the process select download_mode to be all_files else if you want only the final pdb file select final_pdb_only

sequence = "GE(CGU)(CGU)YQ(CGU)ML(CGU)NLR(CGU)AEVKKNA"  # @param {type:"string"}
job_name  = "test_seq"                   # @param {type:"string"}
# new shit
download_mode = "all_files"  # @param ["final_pdb_only", "all_files"]


import os, zipfile, shutil
from pathlib import Path
from google.colab import files
from IPython.utils import io
from IPython.display import HTML, display

%cd /content/batch_peptide_pipeline

DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'

# new shit
# Ensure Drive folders exist before main.py tries to write there
import os
os.makedirs(f'{DRIVE_BASE}/converged', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/exploded', exist_ok=True)

print(f"🚀 Starting job : {job_name}")
print(f"🧬 Sequence     : {sequence}\n")
print("⏳ Running pipeline (this will take a few minutes)...")

# ── Catch the messy pipeline output ──
with io.capture_output() as captured:
    # ── FIX: override Colab's matplotlib_inline backend so conda envs don't crash ──
    !MPLBACKEND=Agg python main.py --sequence "$sequence" --json modifications.json --job_name "$job_name"

print(f"\n✅ Job '{job_name}' complete!")

# ── Display the clickable dropdown for the pipeline logs ──
display(HTML(f"""
<details>
    <summary style="cursor: pointer; color: #1a73e8; font-weight: bold; padding: 10px; border: 1px solid #1a73e8; border-radius: 5px; width: fit-content;">
        📊 Click here to view detailed pipeline logs
    </summary>
    <div style="background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; margin-top: 10px; max-height: 400px; overflow-y: auto;">
        <pre style="white-space: pre-wrap; word-wrap: break-word;">{captured.stdout}</pre>
        <pre style="color: red; white-space: pre-wrap; word-wrap: break-word;">{captured.stderr}</pre>
    </div>
</details>
"""))


# new shit
import time
time.sleep(10)  # Give Drive a moment to sync the folder


# ── Find the output folder on Drive (converged or exploded) and zip+download ──
job_folder = None
for subfolder in ['converged', 'exploded']:
    candidate = Path(DRIVE_BASE) / subfolder / job_name
    if candidate.exists():
        job_folder = candidate
        print(f"\n📂 Output saved to Drive under: {subfolder}/{job_name}/")
        break

if job_folder:
    if download_mode == "final_pdb_only":
        pdb_src = job_folder / 'final_minimized.pdb'
        if pdb_src.exists():
            local_pdb = f'/content/{job_name}_final_minimized.pdb'
            shutil.copy(str(pdb_src), local_pdb)
            print(f"⬇️  Downloading {job_name}_final_minimized.pdb...")
            files.download(local_pdb)
        else:
            print("⚠️  final_minimized.pdb not found — job may have exploded.")
    else:
        zip_path = f'/content/{job_name}_output.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, filenames in os.walk(job_folder):
                for fname in filenames:
                    fpath = os.path.join(root, fname)
                    zf.write(fpath, os.path.relpath(fpath, str(job_folder.parent)))
        print(f"⬇️  Downloading {job_name}_output.zip...")
        files.download(zip_path)
else:
    print("⚠️  Output folder not found on Drive — pipeline may have crashed before saving.")

/content/batch_peptide_pipeline
🚀 Starting job : test_ptm
🧬 Sequence     : KETA{ptm:AIB}AAKFERQHLDS

⏳ Running pipeline (this will take a few minutes)...

✅ Job 'test_ptm' complete!



📂 Output saved to Drive under: converged/test_ptm/
⬇️  Downloading test_ptm_output.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title ▶️ Batch Prediction from FASTA (2 sequences in parallel)
# @markdown ⚠️  **This cell wipes all previous prediction output from Drive before starting.**
# @markdown Upload your .fasta file when prompted.
# @markdown Output + CSV will be auto-downloaded as a zip when done.

import os, shutil, json, zipfile, subprocess
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from google.colab import files
from IPython.utils import io
from IPython.display import HTML, display

REPO_DIR   = '/content/batch_peptide_pipeline'
DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)

# new shit
download_mode = "final_pdb_only"  # @param ["final_pdb_only", "all_files"]

%cd /content/batch_peptide_pipeline

# ── FIX: override Colab's matplotlib_inline backend so conda envs don't crash ──
_clean_env = {**os.environ, 'MPLBACKEND': 'Agg'}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 — Wipe previous output (Keep visible for safety)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("🧹 Clearing previous prediction output from Drive...")
for folder in ['converged', 'exploded']:
    p = drive_path / folder
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(exist_ok=True)

csv_path = drive_path / 'batch_summary.csv'
if csv_path.exists():
    csv_path.unlink()
print("✅ Previous output cleared\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 — Upload and parse FASTA (Must be visible)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("👇 Please upload your .fasta file below:")
uploaded = files.upload()

if len(uploaded) == 0:
    print("\n❌ No file uploaded. Please re-run this cell.")
else:
    fasta_file = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {fasta_file}")

    sequences = {}
    with open(fasta_file) as f:
        current_id = None
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                current_id = line[1:].strip()
            elif current_id and line:
                sequences[current_id] = sequences.get(current_id, '') + line

    print(f"Found {len(sequences)} sequences. Running 2 in parallel...")
    print("⏳ Processing pipeline... (Check dropdown below for progress)")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 3 & 4 — Run in parallel (CAPTURED)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    def run_one_sequence(seq_id, sequence):
        try:
            cmd = ['python', 'main.py', '--sequence', sequence, '--json', 'modifications.json', '--job_name', seq_id]
            result = subprocess.run(cmd, cwd=REPO_DIR, env=_clean_env, capture_output=False, text=True)
            if result.returncode == 0:
                print(f"  ✅ {seq_id} — Done")
            else:
                print(f"  ❌ {seq_id} — Failed (exit code {result.returncode})")
            return result.returncode == 0
        except Exception as e:
            print(f"  ❌ {seq_id} — Error: {e}")
            return False

    # Capture the messy parallel logs
    with io.capture_output() as captured:
        print("=" * 60)
        print("  Running pipeline...")
        print("=" * 60)
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = {seq_id: executor.submit(run_one_sequence, seq_id, seq)
                       for seq_id, seq in sequences.items()}
        job_success = {seq_id: future.result() for seq_id, future in futures.items()}

    # Display the logs dropdown
    display(HTML(f"""
    <details>
        <summary style="cursor: pointer; color: #1a73e8; font-weight: bold; padding: 10px; border: 1px solid #1a73e8; border-radius: 5px; width: fit-content;">
            📊 View Parallel Processing Logs
        </summary>
        <div style="background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; margin-top: 10px; max-height: 300px; overflow-y: auto;">
            <pre style="white-space: pre-wrap; word-wrap: break-word;">{captured.stdout}</pre>
        </div>
    </details>
    """))

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 5 & 6 — Collect Results & Save CSV (Visible)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    all_results = []
    for seq_id in sequences.keys():
        status_json = None
        job_folder  = None
        for subfolder in ['converged', 'exploded']:
            candidate = drive_path / subfolder / seq_id / 'minimize_status.json'
            if candidate.exists():
                status_json = candidate
                job_folder  = subfolder
                break

        if status_json:
            with open(status_json) as f:
                status = json.load(f)
            status['name']   = seq_id
            status['folder'] = job_folder
            all_results.append(status)
        else:
            all_results.append({'name': seq_id, 'folder': 'failed', 'converged': False, 'exploded': False})

    if all_results:
        df = pd.DataFrame(all_results)
        df.to_csv(str(csv_path), index=False)
        n_conv = int(df['converged'].sum())
        n_expl = int(df['exploded'].sum())

        print(f"\n{'=' * 60}")
        print(f"  BATCH COMPLETE — ✅ {n_conv} Converged | 💥 {n_expl} Exploded")
        print(f"{'=' * 60}")
        print(f"📄 Summary saved to: {csv_path}")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # STEP 7 — Zip & Download (Visible)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    zip_path = '/content/batch_output.zip'
    print(f"\n📦 Zipping output for download...")

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for subfolder in ['converged', 'exploded']:
            p = drive_path / subfolder
            if p.exists():
                for job_dir in p.iterdir():
                    if not job_dir.is_dir(): continue
                    if download_mode == "final_pdb_only":
                        pdb = job_dir / 'final_minimized.pdb'
                        if pdb.exists(): zf.write(str(pdb), f"{subfolder}/{job_dir.name}/final_minimized.pdb")
                    else:
                        for root, dirs, filenames in os.walk(job_dir):
                            for fname in filenames:
                                fpath = os.path.join(root, fname)
                                zf.write(fpath, os.path.relpath(fpath, str(drive_path)))
        if csv_path.exists():
            zf.write(str(csv_path), 'batch_summary.csv')

    print(f"⬇️  Downloading batch_output.zip ({download_mode})...")
    files.download(zip_path)
    print("\n🎉 Done!")

/content/batch_peptide_pipeline
🧹 Clearing previous prediction output from Drive...
✅ Previous output cleared

👇 Please upload your .fasta file below:


Saving test.fasta to test (1).fasta

✅ Uploaded: test (1).fasta
Found 2 sequences. Running 2 in parallel...
⏳ Processing pipeline... (Check dropdown below for progress)



  BATCH COMPLETE — ✅ 2 Converged | 💥 0 Exploded
📄 Summary saved to: /content/drive/MyDrive/try_pipeline/batch_summary.csv

📦 Zipping output for download...
⬇️  Downloading batch_output.zip (final_pdb_only)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Done!


In [ ]:
# @title 🛟 Recovery Cell — Run only if batch crashed midway
# @markdown Scans Drive for all jobs that did complete, builds a recovered CSV,
# @markdown and auto-downloads everything as a zip.
# @markdown (Does NOT wipe anything — safe to run at any time.)

import os, json, zipfile
import pandas as pd
from pathlib import Path
from google.colab import files

DRIVE_BASE = '/content/drive/MyDrive/try_pipeline'
drive_path = Path(DRIVE_BASE)
csv_path   = drive_path / 'batch_summary_recovered.csv'

print("🔍 Scanning Drive for completed jobs...\n")

all_results = []

for subfolder in ['converged', 'exploded']:
    folder_path = drive_path / subfolder
    if not folder_path.exists():
        continue
    for job_dir in sorted(folder_path.iterdir()):
        if not job_dir.is_dir():
            continue
        status_file = job_dir / 'minimize_status.json'
        if status_file.exists():
            with open(status_file) as f:
                status = json.load(f)
            status['name']   = job_dir.name
            status['folder'] = subfolder
            all_results.append(status)
            if status.get('exploded'):
                print(f"  💥 {job_dir.name}: EXPLODED at step {status['steps']}")
            elif status.get('converged'):
                print(f"  ✅ {job_dir.name}: Converged in {status['steps']} steps, "
                      f"RMSD={status['rmsd_heavy']:.3f} Å")
            else:
                print(f"  ⚠️  {job_dir.name}: Max steps ({status['steps']})")
        else:
            print(f"  ❌ {job_dir.name}: Folder exists but no status file (incomplete job)")

if not all_results:
    print("❌ No completed jobs found on Drive.")
else:
    df = pd.DataFrame(all_results)
    col_order  = ['name', 'folder', 'converged', 'exploded', 'n_atoms',
                  'steps', 'e0', 'e1', 'delta_e', 'time_sec',
                  'rmsd_all', 'rmsd_heavy', 'max_disp']
    extra_cols = [c for c in df.columns if c not in col_order]
    df = df[[c for c in col_order if c in df.columns] + extra_cols]
    df.to_csv(str(csv_path), index=False)

    n_conv = int(df['converged'].sum())
    n_expl = int(df['exploded'].sum())

    print(f"\n{'=' * 60}")
    print(f"  RECOVERED — {len(df)} completed jobs found")
    print(f"{'=' * 60}")
    print(f"  ✅ Converged : {n_conv}")
    print(f"  💥 Exploded  : {n_expl}")
    print(f"  📄 CSV saved : {csv_path}")

    zip_path = '/content/batch_recovered.zip'
    print(f"\n📦 Zipping recovered output...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for subfolder in ['converged', 'exploded']:
            p = drive_path / subfolder
            if p.exists():
                for root, dirs, filenames in os.walk(p):
                    for fname in filenames:
                        fpath = os.path.join(root, fname)
                        zf.write(fpath, os.path.relpath(fpath, str(drive_path)))
        zf.write(str(csv_path), 'batch_summary_recovered.csv')

    print("⬇️  Downloading batch_recovered.zip...")
    files.download(zip_path)
    print("\n🎉 Recovery complete!")
